In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

from src.data.load_data import load_raw_data, basic_data_summary
from src.features.preprocess import (
    ZERO_AS_MISSING_COLUMNS,
    count_zero_values,
    preprocess_pima_data,
)

print("Project root:", project_root)

raw_df = load_raw_data()
summary = basic_data_summary(raw_df)

summary

Project root: /Users/galaxygab/Desktop/Recurrent-Neural-Networks-for-Disease-Progression-Prediction


FileNotFoundError: Could not find dataset at: data/raw/Pima_Diabetes.csv

In [ ]:
print("Shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())

print("\nFirst 5 rows:")
display(raw_df.head())

print("\nClass distribution:")
print(raw_df["Class"].value_counts())

print("\nMissing values:")
print(raw_df.isna().sum())

print("\nDescriptive statistics:")
display(raw_df.describe())

In [ ]:
zero_counts = count_zero_values(raw_df, ZERO_AS_MISSING_COLUMNS)
zero_counts

### Note on Data Cleaning
In the Pima dataset, several features contain zero values that are medically implausible in real patients, including glucose, blood pressure, skin fold thickness, insulin, and BMI. These are treated as missing values and imputed using the median so the pipeline remains clinically reasonable and robust.

In [ ]:
clean_df, zero_counts_before, missing_after_zero_replacement = preprocess_pima_data()

print("Zero counts before cleaning:")
print(zero_counts_before)

print("\nMissing values immediately after zero replacement:")
print(missing_after_zero_replacement)

print("\nMissing values after imputation:")
print(clean_df.isna().sum())

display(clean_df.head())

In [ ]:
fig_path = project_root / "outputs" / "figures"
fig_path.mkdir(parents=True, exist_ok=True)

class_counts = raw_df["Class"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(["No Diabetes (0)", "Diabetes (1)"], class_counts.values)
plt.title("Class Distribution in the Pima Diabetes Dataset")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.savefig(fig_path / "class_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
feature_columns = [col for col in clean_df.columns if col != "Class"]

for feature in feature_columns:
    plt.figure(figsize=(8, 5))
    plt.hist(clean_df[feature], bins=25)
    plt.title(f"{feature} Distribution After Cleaning")
    plt.xlabel(feature)
    plt.ylabel("Frequency")
    plt.tight_layout()

    safe_name = feature.lower().replace(" ", "_").replace("-", "_")
    plt.savefig(fig_path / f"hist_{safe_name}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
for feature in feature_columns:
    plt.figure(figsize=(8, 5))

    data_0 = clean_df[clean_df["Class"] == 0][feature]
    data_1 = clean_df[clean_df["Class"] == 1][feature]

    plt.boxplot([data_0, data_1], tick_labels=["Class 0", "Class 1"])
    plt.title(f"{feature} by Diabetes Class")
    plt.ylabel(feature)
    plt.tight_layout()

    safe_name = feature.lower().replace(" ", "_").replace("-", "_")
    plt.savefig(fig_path / f"boxplot_{safe_name}_by_class.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
corr = clean_df.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
plt.imshow(corr, interpolation="nearest", aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(fig_path / "correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
summary_table = clean_df.describe().T
summary_table["missing_after_cleaning"] = clean_df.isna().sum()
summary_table["class_0_mean"] = [
    clean_df[clean_df["Class"] == 0][col].mean() if col != "Class" else clean_df[clean_df["Class"] == 0][col].mean()
    for col in summary_table.index
]
summary_table["class_1_mean"] = [
    clean_df[clean_df["Class"] == 1][col].mean() if col != "Class" else clean_df[clean_df["Class"] == 1][col].mean()
    for col in summary_table.index
]

table_path = project_root / "outputs" / "tables"
table_path.mkdir(parents=True, exist_ok=True)
summary_table.to_csv(table_path / "eda_summary_table.csv")

summary_table